# Perbaikan kandidat GPU: pecah proyeksi GLU sebelum convolution

Run GPU pertama berhasil, tetapi batch berikutnya mengalami fatal assertion MuQ 3000-frame (`mps.permute`, 1024 vs 2048 kanal). Jangan memakai project GPU v1 untuk pengujian lanjutan. Keberhasilan satu run belum menunjukkan stabilitas.

Kandidat ini mempertahankan **payload bobot palette6 yang sama**. Sepuluh proyeksi GLU pada setiap graph diubah menjadi dua convolution 1024 kanal sebelum sigmoid, menggantikan convolution 2048 kanal lalu memotong output. Tidak ada training atau kuantisasi ulang. Perubahan ini merupakan hipotesis perbaikan compiler; hasil iPhone masih perlu diuji.

Buka notebook dari root **SongMap** di VS Code. Kernel boleh dari Section-Annotations; semua subprocess menggunakan executable venv tersebut secara eksplisit. Persiapan menghasilkan project benchmark terpisah dan memeriksa SDK Swift serta dua lagu PCM di CPU Mac. Aset/routing SongMap tidak ditimpa. Hasil full validation 32 lagu sebelumnya adalah bukti untuk graph induk, belum full validation graph perbaikan ini.

Pemeriksaan pooled feature menggunakan aturan elementwise ekspor encoder: `abs(actual-reference) <= 1e-4 + 1e-4*abs(reference)`. Probabilitas base dan akhir tetap memakai selisih absolut ≤`1e-4`, dan label akhir harus 100% sama. Selisih maksimum, posisi/nilai terburuk dan pelanggaran toleransi disimpan dalam `mac-preflight.json`, termasuk ketika ditolak. Warning scikit-learn dan deprecation Accelerate tidak menyebabkan penghentian parity.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib,json,subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/repair_encoder_glu.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Jangan resolve symlink executable venv.
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
reports=[]
for p in (ROOT/'outputs.local').glob('encoder-pipeline-quality-*/validation-all.json'):
    r=json.loads(p.read_text())
    if (r.get('full_validation') and r.get('section_agreement')==1
            and r.get('comparison_to_current_pipeline',{}).get('no_observed_drop')):reports.append(p)
assert reports, 'Report full validation encoder_pipeline_quality.ipynb belum tersedia'
QUALITY=max(reports,key=lambda p:p.stat().st_mtime).resolve()
selection=json.loads((QUALITY.parent/'selection.json').read_text())
SOURCES={}
for role,expected in selection['encoder_exports'].items():
    candidates=(ROOT/'outputs.local').glob('encoder-acceleration-*/'+role+'/export-report.json')
    SOURCES[role]=next((p.resolve() for p in candidates if sha(p)==expected),None)
    assert SOURCES[role], f'Sumber export {role} yang cocok belum ditemukan'
# Gunakan ulang repair selesai yang berasal dari script dan export yang sama; jangan kuantisasi ulang.
expected_receipt={'script':sha(ROOT/'scripts/repair_encoder_glu.py'),
                  'parents':{role:sha(path) for role,path in SOURCES.items()}}
completed=[]
for path in (ROOT/'outputs.local').glob('encoder-glu-repair-*/repair-report.json'):
    receipt=path.parent/'notebook-source.json'
    if receipt.is_file() and json.loads(receipt.read_text())==expected_receipt:
        proof=json.loads(path.read_text())
        if proof.get('parent_exports')==expected_receipt['parents'] and set(proof.get('roles',{}))=={'musicfm','muq'}:
            completed.append(path.parent)
OUTPUT=(max(completed,key=lambda p:p.stat().st_mtime).resolve() if completed else
        ROOT/'outputs.local'/('encoder-glu-repair-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')))
print('Quality induk:',QUALITY)
print('Sumber:',SOURCES)
print('Output perbaikan:',OUTPUT)

def run_logged(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen([str(v) for v in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Gagal ({status}); lihat {log}. Jangan lanjut build. Untuk ulang repair yang belum selesai, jalankan cell setup agar OUTPUT baru.')


In [ ]:
# Empat graph: MusicFM/MuQ x 750/3000. CPU parity asli -> hasil perbaikan, tiga prediction per graph.
REPAIR=OUTPUT/'repair-report.json'
source_receipt={'script':sha(ROOT/'scripts/repair_encoder_glu.py'),
                'parents':{role:sha(path) for role,path in SOURCES.items()}}
receipt=OUTPUT/'notebook-source.json'
if REPAIR.is_file() and receipt.is_file():
    assert json.loads(receipt.read_text())==source_receipt, 'Sumber berubah; jalankan cell setup untuk OUTPUT baru'
    print('Repair yang selesai digunakan kembali:',REPAIR)
else:
    run_logged([RUN_PYTHON,'-u',ROOT/'scripts/repair_encoder_glu.py',
        '--musicfm-export',SOURCES['musicfm'],'--muq-export',SOURCES['muq'],
        '--output-dir',OUTPUT],Path(str(OUTPUT)+'.repair.log'))
    receipt.write_text(json.dumps(source_receipt,indent=2)+'\n')
proof=json.loads(REPAIR.read_text())
assert set(proof['roles'])=={'musicfm','muq'} and proof['weights_requantized'] is False
for role,entry in proof['roles'].items():
    path=OUTPUT/entry['report'];assert sha(path)==entry['sha256']
    report=json.loads(path.read_text())
    for check in report['checks']:
        print(role,check['frames'],'payload sama:',check['compressed_payloads_unchanged'],
              'convolution dipecah:',len(check['changed_convolutions']),
              'CPU parity:',all(v['passed'] for v in check['cpu_checks']))
print('Belum membuktikan stabilitas GPU iPhone.')


In [ ]:
# Setiap pemanggilan cell ini memakai folder project baru; hasil repair di atas tetap dapat dipakai.
PHONE=OUTPUT/('iphone-project-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
run_logged([RUN_PYTHON,'-u',ROOT/'scripts/prepare_encoder_pipeline_iphone.py',
    '--songstruct-root',SONGSTRUCT_ROOT,'--songmap-root',ROOT,'--quality-report',QUALITY,
    '--graph-repair',REPAIR,'--output-dir',PHONE],Path(str(PHONE)+'.prepare.log'))
preparation=json.loads((PHONE/'preparation.json').read_text())
assert preparation['static_graph_policy']=='split-before-conv-v2'
assert preparation['native_typecheck_passed'] and preparation['mac_pcm_pipeline_passed']
assert preparation['graph_repair_sha256']==sha(REPAIR)
print('SDK typecheck dan PCM CPU Mac lolos. Project:',PHONE/'Benchmark.xcodeproj')
for check in preparation['checks']:print(check)
subprocess.run(['open',str(PHONE/'Benchmark.xcodeproj')],check=True)


## Uji iPhone pada project baru

Project menghasilkan aplikasi **GLU Repair** dengan bundle ID terpisah agar report lama tidak tercampur. Pilih Signing Team dan iPhone fisik. Tidak perlu launch argument. Mulai saat perangkat tidak panas; pertahankan kondisi perangkat yang sama antar-run.

1. Pilih **CPU + GPU candidate**, lagu pertama, lalu **Run selected track**. Pastikan run selesai dan report menyatakan `static_graph_policy=split-before-conv-v2`, label sesuai referensi Mac, serta tidak ada NaN/Inf.
2. Tutup aplikasi sepenuhnya, buka ulang, pilih **CPU + GPU candidate**, lalu **Run 3 passes over all tracks**. Tunggu dua lagu x tiga pass dalam satu proses. Ini melatih pergantian shape, partial CPU, window 120 detik dan head chunk pada lagu panjang.
3. Bagikan **batch JSON**, `repair-report.json`, dan `preparation.json`. Jika crash, buka ulang lalu segera bagikan **Share last recorded stage** sebelum memulai run lain. Report mencatat model, shape/stride input, window dan mode sebelum prediction. Fatal assertion tidak dapat ditangkap Swift `catch`.

Jika single run gagal, berhenti sebelum batch; kirim log dan checkpoint. Project v1 tanpa perbaikan sekarang hanya menawarkan CPU. GPU kandidat tetap belum disetujui untuk integrasi produksi. Jika stress test lolos, lanjutkan [encoder_glu_quality.ipynb](encoder_glu_quality.ipynb) untuk full validation graph perbaikan. Setelah kualitas diperiksa, lakukan perbandingan waktu CPU/GPU dengan urutan seimbang dan penataan aset agar bobot statis/dinamis tidak menggandakan ukuran aplikasi.

Angka waktu berasal dari PCM -> encoder -> head -> konteks dengan boundary yang diberikan; decode MP3, MusicUnderstanding, beat, chord dan lirik tidak termasuk. Run terdahulu 51.29s CPU vs 39.85s GPU juga berbeda waktu load/cache (12.89s vs 2.40s); itu belum menunjukkan speedup inferensi 1.29x. Snapshot memori bukan pengukuran peak.
